In [2]:
%load_ext autoreload
%autoreload 2

import mujoco
from mujoco import MjsBody
import numpy as np
import PIL.Image

from swarmbots.unit import init_unit
from swarmbots.mujoco_utils import quat_z2vec, qpos_indices_for_prefix, ctrl_indices_for_prefix, body_ids_for_prefix
from rendering import display_video, display_image


In [2]:
RENDER_WIDTH = 640
RENDER_HEIGHT = 480


In [4]:

spec = mujoco.MjSpec()
worldbody: MjsBody = spec.worldbody

unit = init_unit(
    body_radius=0.1,
    leg_length=0.2,
    leg_radius=0.025,
    hinge_range=np.pi / 4
)
unit.add_joint(type=mujoco.mjtJoint.mjJNT_FREE)

worldbody.add_frame(pos=[0, 0, 2]).attach_body(unit, 'Unit1--', '')  # , quat=quat_z2vec([1, 1, 0])

unit = init_unit(
    body_radius=0.1,
    leg_length=0.2,
    leg_radius=0.025,
    hinge_range=np.pi / 4
)
unit.add_joint(type=mujoco.mjtJoint.mjJNT_FREE)

worldbody.add_frame(pos=[0, 0.6, 2]).attach_body(unit, 'Unit2--', '')


worldbody.add_geom(
    type=mujoco.mjtGeom.mjGEOM_PLANE,
    size=[10, 10, 0.1],
    rgba=[0.2, 0.3, 0.4, 1],
    pos=[0, 0, 0]
)

worldbody.add_light(pos=[0, 0, 6], dir=[0, 0, -1])
worldbody.add_light(pos=[2, 2, 6], dir=[-1, -1, -1])


eq = spec.add_equality(
    name="site_weld",
    type=mujoco.mjtEq.mjEQ_WELD,        # or mjEQ_CONNECT
    objtype=mujoco.mjtObj.mjOBJ_BODY,  # tells MuJoCo the objs are sites
    name1="Unit1--limb_yp-tip",
    name2="Unit2--limb_yn-tip",
    
    #   data[0:7]  = relpose (3 pos + 4 quat)
    #   data[7:10] = anchor  (3)
    #   data[10]   = torquescale
    # data=[0,0,0,0,1,0,0,  0,0,0,  10.0]
)
eq.data[10] = 50

worldbody.add_camera(pos=[-3, 0, 3], mode=mujoco.mjtCamLight.mjCAMLIGHT_TARGETBODY, targetbody=unit.name)

model = spec.compile()
data = mujoco.MjData(model)

renderer = mujoco.Renderer(model, height=RENDER_HEIGHT, width=RENDER_WIDTH)

opt = mujoco.MjvOption() 
opt.sitegroup[:] = 1    
opt.frame = mujoco.mjtFrame.mjFRAME_BODY 
# opt.label = mujoco.mjtLabel.mjLABEL_SITE  

print(f"Model compiled. nq={model.nq}, nv={model.nv}")

# mujoco.mj_step(model, data)

Model compiled. nq=38, nv=36


In [5]:
quat_z2vec([0, 0, -1])

array([0., 1., 0., 0.])

In [6]:
data.eq_active

array([1], dtype=uint8)

In [7]:
model.body_jntnum[mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'Unit1--main_body')]

np.int32(1)

In [8]:
mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, 'Unit1--main_body')

1

In [9]:
print(qpos_indices_for_prefix(model, 'Unit2'))
print()
print(ctrl_indices_for_prefix(model, 'Unit2'))
print()
print(body_ids_for_prefix(model, 'Unit1'))

[19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37]

[12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25]


In [10]:
model.body(2).name

'Unit1--limb_root_xp'

In [11]:
print(spec.to_xml())

<mujoco model="MuJoCo Model">
  <compiler angle="radian"/>

  <default>
    <default class="Unit1--main"/>
    <default class="Unit2--main"/>
  </default>

  <worldbody>
    <geom size="10 10 0.1" type="plane" rgba="0.2 0.3 0.4 1"/>
    <camera target="Unit2--main_body" pos="-3 0 3" mode="targetbody"/>
    <light pos="0 0 6" dir="0 0 -1"/>
    <light pos="2 2 6" dir="-0.57735 -0.57735 -0.57735"/>
    <body name="Unit1--main_body" pos="0 0 2">
      <joint type="free"/>
      <geom size="0.1" rgba="0.75 0 0 0.1"/>
      <body name="Unit1--limb_root_xp" pos="0.1 0 0" quat="0.707107 0 0.707107 0">
        <body name="Unit1--limb_xp">
          <joint name="Unit1--limb_xp-hinge1z" pos="0 0 0" axis="0 0 1"/>
          <geom size="0.025 0.01" pos="0 0 0.01" quat="0 1 0 0" type="cylinder" rgba="0 0 0 1"/>
          <body name="Unit1--limb_xp-seg2" pos="0 0 0.02">
            <joint name="Unit1--limb_xp-hinge2x" pos="0 0 0" axis="1 0 0" range="-0.785398 0.785398"/>
            <geom size="0.02

In [12]:
quat_z2vec([0, 0, 1])

array([1., 0., 0., 0.])

In [13]:

# mujoco.mj_resetData(model, data)

# while data.time < 1:
#     mujoco.mj_step(model, data)

# renderer.update_scene(data, camera=-1)
# display_image(renderer.render().copy())

In [15]:
# Simple simulation loop for smoke testing
DURATION = 5.0  # seconds
FRAMERATE = 30  # Hz
frames = []

mujoco.mj_resetData(model, data)

data.ctrl[:] = 0.1

while data.time < DURATION:
    mujoco.mj_step(model, data)
    if len(frames) < data.time * FRAMERATE:
        renderer.update_scene(data, camera=0, scene_option=opt)
        frames.append(renderer.render().copy())

print(f"Simulated {len(frames)} frames")
display_video(frames, FRAMERATE)


Simulated 151 frames


In [ ]:
mujoco.MjSpec().compiler

In [ ]:
data.site(0)

In [7]:
%load_ext autoreload
%autoreload 2

import mujoco
from swarmbots.envs.swarm_bot_env import SwarmBotEnv
from swarmbots.swarm.simple_swarm import SimpleSwarm
from swarmbots.scenarios.obstacle_dungeon_scenario import ObstacleDungeonScenario
from rendering import display_video

# Initialize swarm and scenario
swarm = SimpleSwarm()
scenario = ObstacleDungeonScenario(swarm)

opt = mujoco.MjvOption() 
opt.sitegroup[:] = 1    
opt.frame = mujoco.mjtFrame.mjFRAME_BODY 

# Initialize environment
env = SwarmBotEnv(
    scenario=scenario,
    render_mode="rgb_array",
    width=640,
    height=480,
    camera=0,
    scene_option=opt,
)

obs, info = env.reset()

frames = []
FRAMERATE = 30
done = False

while not done:
    # Random action
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    
    # Record frame if it's time
    if len(frames) < env.data.time * FRAMERATE:
        frame = env.render()
        if frame is not None:
            frames.append(frame)
        
    done = terminated or truncated

env.close()
print(f"Recorded {len(frames)} frames")
display_video(frames, FRAMERATE)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Recorded 151 frames


In [43]:
env.data.qvel.shape

(36,)